# Code walkthrough: guardrailed multi-agent system

This notebook shows the key code of the project in one place. It also runs small live demonstrations of each part.

* It needs **no key** and **no model service**. The demonstrations use stand-in workers, so every result is predictable.
* It runs in VS Code (choose the `.venv` kernel) or in Google Colab, in about a minute.
* The code shown below is read straight from the project files in `src/guardrailed_mas/`, so what you see is exactly what runs.

The live run with real models, the attempt cap in action and the memory surviving a restart are in the companion notebook `run_in_colab.ipynb`.



## 1. Setup

In Colab this cell downloads the project from GitHub and installs the light libraries (LangGraph, Pydantic, PyYAML, pytest). On your Mac it simply uses the project folder you already have.

Change `GITHUB_USER` if your GitHub username is different.

In [1]:
import os, sys, inspect, textwrap

GITHUB_USER = "mapanya"
REPO_NAME = "guardrailed-multi-agent-system"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_NAME):
        !git clone -q https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    os.chdir(f"/content/{REPO_NAME}")
    !pip install -q -r requirements-dev.txt
else:
    # Running locally: move from the notebooks folder up to the project folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.path.abspath("src"))   # lets Python find the project code
print("Project folder:", os.getcwd())

Project folder: /Users/tawengwat/Desktop/AI Projects/Github/GitHub/guardrailed-multi-agent-system


In [2]:
def show(obj):
    """Print the real source code of a class or function, with its file and line number."""
    lines, start = inspect.getsourcelines(obj)
    path = os.path.relpath(inspect.getsourcefile(obj))
    print(f"# {path}, line {start}")
    print(textwrap.dedent("".join(lines)))

print("Ready.")

Ready.


## 2. Settings: the numbers that control the guardrails

All limits live in `config/settings.yaml`, so they can change without touching code. The cap is `max_executions: 3`.

In [3]:
from guardrailed_mas.config import load_settings

settings = load_settings()
g = settings.guardrails
print("max_executions :", g.max_executions)
print("kill_switch    :", g.kill_switch)
print("recursion_limit:", g.recursion_limit)
print("worker model   :", settings.models.worker_model)
print("helper model   :", settings.models.helper_model)

max_executions : 3
kill_switch    : False
recursion_limit: 25
worker model   : openai/gpt-oss-120b
helper model   : openai/gpt-oss-20b


## 3. Structured output: the strict forms (PydanticAI schemas)

Every worker must answer in a fixed shape. The same classes are given to PydanticAI as its required `output_type`.

Key rules: unknown fields are rejected (`extra="forbid"`), values have limits. A rejection must always list the changes needed.

In [4]:
from guardrailed_mas import schemas

show(schemas.StrictModel)
show(schemas.ReviewerOutput)
show(schemas.FallbackResponse)

# src/guardrailed_mas/schemas.py, line 24
class StrictModel(BaseModel):
    """Base class: reject unknown fields and trim stray spaces."""

    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

# src/guardrailed_mas/schemas.py, line 78
class ReviewerOutput(StrictModel):
    """The Reviewer's decision on the Executor's work."""

    approved: bool
    score: int = Field(ge=0, le=10, description="Overall quality from 0 to 10.")
    issues: list[str] = Field(default_factory=list, max_length=10)
    required_changes: list[str] = Field(default_factory=list, max_length=10)

    @model_validator(mode="after")
    def rejection_needs_reasons(self) -> "ReviewerOutput":
        if not self.approved and not self.required_changes:
            raise ValueError("A rejection must list at least one required change.")
        return self

# src/guardrailed_mas/schemas.py, line 116
class FallbackResponse(StrictModel):
    """The fixed, safe answer returned whenever a guardrail stops 

**Try it.** A good review passes; a sneaky extra field, a rejection with no reasons and an edited fallback message are all refused.

In [5]:
from pydantic import ValidationError
from guardrailed_mas.schemas import ReviewerOutput, FallbackResponse

print("Good review:", ReviewerOutput(approved=True, score=9).model_dump())

tests = {
    "extra field": lambda: ReviewerOutput(approved=True, score=9, run_shell_command="rm -rf /"),
    "rejection without reasons": lambda: ReviewerOutput(approved=False, score=2),
    "score out of range": lambda: ReviewerOutput(approved=True, score=42),
    "edited fallback message": lambda: FallbackResponse(reason="kill_switch", task="x", attempts_used=0, message="All good!"),
}
for name, attempt in tests.items():
    try:
        attempt()
        print(f"{name:28} ACCEPTED (unexpected)")
    except ValidationError as err:
        print(f"{name:28} REJECTED: {err.errors()[0]['msg']}")

Good review: {'approved': True, 'score': 9, 'issues': [], 'required_changes': []}
extra field                  REJECTED: Extra inputs are not permitted
rejection without reasons    REJECTED: Value error, A rejection must list at least one required change.
score out of range           REJECTED: Input should be less than or equal to 10
edited fallback message      REJECTED: Value error, The fallback message cannot be changed at run time.


### How replies are checked

Layer 1 pulls the JSON out of a reply and validates it directly, with no model call. Only if that fails does layer 2 ask the PydanticAI checker (an agent whose `output_type` is the schema) to repair it, with a fixed number of retries.

In [6]:
from guardrailed_mas import validation

show(validation.PydanticAIChecker)
show(validation.validate_output)

# src/guardrailed_mas/validation.py, line 91
class PydanticAIChecker:
    """Layer 2: a PydanticAI agent that must return an instance of the schema."""

    INSTRUCTIONS = (
        "You convert a worker's reply into the required structured form. "
        "Keep the worker's meaning. Do not add new facts. "
        "Fill every required field and respect every limit in the form."
    )

    def __init__(self, model: Any, retries: int = 2) -> None:
        self._model = model
        self._retries = retries
        self._agents: dict[type[BaseModel], Any] = {}

    def _agent_for(self, schema: type[BaseModel]) -> Any:
        if schema not in self._agents:
            from pydantic_ai import Agent  # imported here so tests do not need it

            self._agents[schema] = Agent(
                self._model,
                output_type=schema,
                instructions=self.INSTRUCTIONS,
                name=f"{schema.__name__}_checker",
                retries={"tools": 1, "output": 

In [7]:
import json
from guardrailed_mas.schemas import PlannerOutput
from guardrailed_mas.validation import validate_output, OutputValidationError

good_plan = {
    "objective": "Contain the phishing incident and protect finance accounts.",
    "steps": [
        {"step_number": 1, "action": "Isolate affected mailboxes", "owner_role": "SOC analyst"},
        {"step_number": 2, "action": "Reset credentials of users who clicked", "owner_role": "Service desk"},
        {"step_number": 3, "action": "Block the sender and link", "owner_role": "Email administrator"},
    ],
    "key_risks": ["Supplier payment fraud"],
}

# A model reply that wraps the JSON in chatter and a code fence
reply = "Here is the plan:\n```json\n" + json.dumps(good_plan) + "\n```\nHope this helps."
result = validate_output(reply, PlannerOutput)
print("Validated by:", result.method, "| steps:", len(result.value.steps))

# A reply with no JSON at all and no checker available
try:
    validate_output("Plan: isolate, reset, block.", PlannerOutput)
except OutputValidationError as err:
    print("Refused:", err)

# The same reply with a stand-in checker (in the real system this is the PydanticAI agent)
stand_in_checker = lambda text, schema: schema.model_validate(good_plan)
print("Repaired by:", validate_output("Plan: isolate, reset, block.", PlannerOutput, stand_in_checker).method)

Direct validation failed, calling the PydanticAI checker: No JSON object found for PlannerOutput.


Validated by: direct | steps: 3
Refused: No JSON object found for PlannerOutput.
Repaired by: pydantic_ai_checker


## 4. The graph state and the counter

The job record that LangGraph passes from step to step. `execution_count` is the counter the guardrail checks. It starts at 0 for every job.

In [8]:
from guardrailed_mas import state

show(state.GraphState)
fresh = state.initial_state("Phishing report", "demo_user", "security_incident", max_executions=3)
print({k: fresh[k] for k in ("execution_count", "attempts_completed", "max_executions", "halt_reason")})

# src/guardrailed_mas/state.py, line 13
class GraphState(TypedDict, total=False):
    # Inputs
    task: str
    user_id: str
    profile: str

    # Short facts recalled from memory (never the full past conversation)
    memory_context: list[str]

    # Worker results, stored as plain dictionaries after validation
    plan: Optional[dict[str, Any]]
    draft: Optional[dict[str, Any]]
    review: Optional[dict[str, Any]]
    feedback: list[str]

    # Guardrail fields
    execution_count: int          # attempt counter checked by the guardrail
    attempts_completed: int       # how many times the Executor actually ran
    max_executions: int           # the cap (3)
    halt_reason: Optional[str]    # why a guardrail stopped the run, if it did

    # Final answer (FinalResult or FallbackResponse as a dictionary)
    status: str
    final: Optional[dict[str, Any]]

{'execution_count': 0, 'attempts_completed': 0, 'max_executions': 3, 'halt_reason': None}


## 5. The guardrail: the cap rule, the conditional edge and the fallback

Plain Python, no model calls. `check_budget` is the conditional edge LangGraph calls before every Executor run.

In [9]:
from guardrailed_mas import guardrails

show(guardrails.budget_exceeded)
show(guardrails.check_budget)
show(guardrails.route_after_review)
show(guardrails.build_fallback)

# src/guardrailed_mas/guardrails.py, line 28
def budget_exceeded(execution_count: int, max_executions: int = MAX_EXECUTIONS) -> bool:
    """The exact rule from the brief: more than the cap means stop."""
    return execution_count > max_executions

# src/guardrailed_mas/guardrails.py, line 43
def check_budget(state: GraphState) -> Literal["executor", "fallback"]:
    """The conditional edge that enforces the cap before every Executor run."""
    if state.get("halt_reason"):
        return "fallback"
    if budget_exceeded(state["execution_count"], state["max_executions"]):
        return "fallback"
    return "executor"

# src/guardrailed_mas/guardrails.py, line 61
def route_after_review(state: GraphState) -> Literal["finalize", "budget_gate", "fallback"]:
    """Approved work is finished. Rejected work goes back through the gate."""
    if state.get("halt_reason"):
        return "fallback"
    review = state.get("review") or {}
    if review.get("approved"):
        return "finalize

**Try it.** Where does the edge send the job for each counter value?

In [10]:
for count in (1, 2, 3, 4):
    job = {"execution_count": count, "max_executions": 3, "halt_reason": None}
    print(f"execution_count = {count}  ->  {guardrails.check_budget(job)}")

print()
print("Kill-switch on  ->", guardrails.check_budget({"execution_count": 1, "max_executions": 3, "halt_reason": "kill_switch"}))

execution_count = 1  ->  executor
execution_count = 2  ->  executor
execution_count = 3  ->  executor
execution_count = 4  ->  fallback

Kill-switch on  -> fallback


## 6. The three workers (CrewAI)

Each worker is a CrewAI agent with a role, a goal and a backstory taken from `config/agents.yaml`. Delegation is switched off so only the graph decides who works next. Each call is one small crew.

In [11]:
from guardrailed_mas import agents
from guardrailed_mas.config import load_profile

show(agents.CrewAIWorkers.__init__)
show(agents.CrewAIWorkers._run)
show(agents.crewai_llm)

profile = load_profile("security_incident")
for name in ("planner", "executor", "reviewer"):
    print(f"{name:9} role: {profile[name]['role']}")

# src/guardrailed_mas/agents.py, line 147
def __init__(self, llm: Any, profile: dict[str, Any], checker: Optional[Checker] = None) -> None:
    from crewai import Agent  # imported here so the tests do not need CrewAI

    self._llm = llm
    self._profile = profile
    self._checker = checker
    self._agents = {
        name: Agent(
            role=profile[name]["role"],
            goal=profile[name]["goal"],
            backstory=profile[name]["backstory"],
            llm=llm,
            allow_delegation=False,   # workers cannot hand work to each other on their own
            max_iter=1,               # one reasoning pass per call; the graph controls retries
            verbose=False,
        )
        for name in ("planner", "executor", "reviewer")
    }

# src/guardrailed_mas/agents.py, line 166
def _run(self, name: str, description: str, expected: str) -> str:
    from crewai import Crew, Process, Task

    with warnings.catch_warnings():
        warnings.simplefilter("igno

## 7. The LangGraph workflow, run end to end

This is the full wiring: nodes, edges and conditional edges. Below it, the graph is run with **stand-in workers** (simple Python objects with the same three methods as the CrewAI workers), so the results are predictable and need no key.

In [12]:
from guardrailed_mas import graph

show(graph.build_graph)

# src/guardrailed_mas/graph.py, line 36
def build_graph(workers: Workers, memory: Optional[MemoryStore] = None, kill_switch: bool = False):
    """Wire the nodes and edges together and return a compiled, runnable graph."""

    # ------------------------------------------------------------------ nodes
    def recall_memory(state: GraphState) -> dict[str, Any]:
        facts: list[str] = []
        if memory is not None:
            try:
                facts = memory.recall(state["user_id"], state["task"])
            except Exception as error:  # memory is best effort
                logger.warning("Memory recall failed, continuing without it: %s", error)
        _event("recall_memory", user_id=state["user_id"], facts_found=len(facts))
        return {"memory_context": facts}

    def planner(state: GraphState) -> dict[str, Any]:
        if guardrails.kill_switch_active(kill_switch):
            _event("planner", halted="kill_switch")
            return {"halt_reason": "kill_switch"}


In [13]:
from guardrailed_mas.graph import build_graph
from guardrailed_mas.schemas import ExecutorOutput, ReviewerOutput
from guardrailed_mas.state import initial_state
from guardrailed_mas.validation import ValidatedOutput
from guardrailed_mas.memory import InMemoryStore

good_draft = {
    "title": "Phishing first-response checklist",
    "summary": "Steps for the first hour after a phishing report.",
    "items": [
        {"order": 1, "action": "Quarantine the email for all users", "owner_role": "Email administrator", "priority": "critical", "time_target": "within 15 minutes"},
        {"order": 2, "action": "Reset passwords for the two users", "owner_role": "Service desk", "priority": "high", "time_target": "within 30 minutes"},
        {"order": 3, "action": "Preserve email headers as evidence", "owner_role": "SOC analyst", "priority": "high", "time_target": "within 1 hour"},
    ],
    "escalation_contacts": ["SOC lead"],
}

class StandInWorkers:
    """Same three methods as the CrewAI workers. The Reviewer approves on a chosen attempt (or never)."""
    def __init__(self, approve_on_attempt=None):
        self.approve_on_attempt = approve_on_attempt
        self.executor_runs = 0
    def plan(self, task, memory_context):
        return ValidatedOutput(PlannerOutput.model_validate(good_plan), "direct")
    def execute(self, task, plan, feedback, memory_context):
        self.executor_runs += 1
        return ValidatedOutput(ExecutorOutput.model_validate(good_draft), "direct")
    def review(self, task, plan, draft):
        ok = self.approve_on_attempt is not None and self.executor_runs >= self.approve_on_attempt
        return ValidatedOutput(ReviewerOutput(approved=ok, score=9 if ok else 3,
                               required_changes=[] if ok else ["Add an evidence preservation step"]), "direct")

def run(workers, kill_switch=False):
    app = build_graph(workers, InMemoryStore(), kill_switch=kill_switch)
    final_state = app.invoke(initial_state("Phishing report", "demo_user", "security_incident", 3),
                             config={"recursion_limit": 25})
    return final_state

print("The graph's steps and arrows (Mermaid text):")
print(build_graph(StandInWorkers()).get_graph().draw_mermaid())

The graph's steps and arrows (Mermaid text):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	recall_memory(recall_memory)
	planner(planner)
	budget_gate(budget_gate)
	executor(executor)
	reviewer(reviewer)
	finalize(finalize)
	fallback(fallback)
	save_memory(save_memory)
	__end__([<p>__end__</p>]):::last
	__start__ --> recall_memory;
	budget_gate -.-> executor;
	budget_gate -.-> fallback;
	executor -.-> budget_gate;
	executor -.-> fallback;
	executor -.-> reviewer;
	fallback --> save_memory;
	finalize --> save_memory;
	planner -.-> budget_gate;
	planner -.-> fallback;
	recall_memory --> planner;
	reviewer -.-> budget_gate;
	reviewer -.-> fallback;
	reviewer -.-> finalize;
	save_memory --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**Run 1: the Reviewer approves on the second attempt.**

In [14]:
w = StandInWorkers(approve_on_attempt=2)
s = run(w)
print("status:", s["status"], "| Executor runs:", w.executor_runs, "| execution_count:", s["execution_count"])

status: approved | Executor runs: 2 | execution_count: 2


**Run 2: the Reviewer always rejects (the guardrail demonstration).** The Executor runs exactly 3 times; when `execution_count` reaches 4 the graph stops and returns the fixed fallback.

In [15]:
w = StandInWorkers(approve_on_attempt=None)
s = run(w)
print("status:", s["status"], "| Executor runs:", w.executor_runs, "| execution_count:", s["execution_count"])
print(json.dumps(s["final"], indent=2))

status: halted | Executor runs: 3 | execution_count: 4
{
  "status": "halted",
  "reason": "execution_budget_exceeded",
  "message": "The automated workflow was stopped by a safety control before it could produce an approved answer. No further automated actions were taken. Please hand this job to a human analyst using your standard escalation procedure.",
  "task": "Phishing report",
  "attempts_used": 3,
  "last_issues": [
    "Add an evidence preservation step"
  ]
}


**Run 3: the kill-switch is on.** No worker is called at all.

In [16]:
w = StandInWorkers(approve_on_attempt=1)
s = run(w, kill_switch=True)
print("status:", s["status"], "| reason:", s["final"]["reason"], "| Executor runs:", w.executor_runs)

status: halted | reason: kill_switch | Executor runs: 0


## 8. Memory (Mem0 on local SQLite and Chroma)

The settings Mem0 is started with: Groq for its language model, a small local text model for search, a Chroma folder as the search index and a **SQLite file** for the history. All of it lives in the `data` folder on disk, which is why it survives a restart.

The live proof (save facts, restart, recall them, count SQLite rows) is Part 3 of `run_in_colab.ipynb`, because Mem0 and its search model need the full install.

In [17]:
from guardrailed_mas import memory
from guardrailed_mas.memory import build_mem0_config

show(memory.Mem0Store)
print(json.dumps(build_mem0_config(settings.memory, settings.models), indent=2))

# src/guardrailed_mas/memory.py, line 56
class Mem0Store:
    """Mem0 memory kept on local disk (SQLite history plus Chroma index)."""

    def __init__(self, memory: MemorySettings, models: ModelSettings) -> None:
        os.environ.setdefault("MEM0_TELEMETRY", "False")   # do not send usage data out
        from mem0 import Memory

        memory.data_dir.mkdir(parents=True, exist_ok=True)
        self._settings = memory
        self._memory = Memory.from_config(build_mem0_config(memory, models))
        logger.info("Memory ready. SQLite history file: %s", memory.sqlite_path)

    @staticmethod
    def _texts(result: Any) -> list[str]:
        items = result.get("results", []) if isinstance(result, dict) else (result or [])
        return [item["memory"] for item in items if isinstance(item, dict) and item.get("memory")]

    def recall(self, user_id: str, query: str) -> list[str]:
        result = self._memory.search(query=query, filters={"user_id": user_id}, top_k=self._settings.ma

## 9. The automated tests and the design write-up

The same checks above (plus more) run as 33 automated tests. The reasons behind the cap of 3 and the schema design are written up in `design_decisions.md`.

In [18]:
# Run the tests with the same Python this notebook uses (safe even when the folder path has spaces)
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(result.stdout[-1500:] or result.stderr[-1500:])

============================= test session starts ==============================
platform darwin -- Python 3.12.14, pytest-9.1.1, pluggy-1.6.0
rootdir: /Users/tawengwat/Desktop/AI Projects/Github/GitHub/guardrailed-multi-agent-system
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.14.1, platformdirs-4.12.0, anyio-4.15.1
collected 33 items

tests/test_guardrailed_mas.py .................................          [100%]

============================== 33 passed in 0.83s ==============================



## Summary

| What the brief asks | Where it is in this notebook |
|---|---|
| 3-node graph: Planner, Executor, Reviewer (LangGraph and CrewAI) | Sections 6 and 7 |
| `execution_count` in the graph state and a conditional edge enforcing the cap | Sections 4 and 5 |
| Stop above 3 and return a safe, defined fallback; kill-switch | Section 7, runs 2 and 3 |
| Output validated against PydanticAI schemas | Section 3 |
| Mem0 on local SQLite | Section 8 (live proof in `run_in_colab.ipynb`) |
| Design decisions write-up | `design_decisions.md` |

